### Learning Instruction Finetuning

In [86]:
%load_ext autoreload
# %load_ext tensorboard
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys
import wandb

In [88]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
env_path = (project_root / "env.env").resolve()
status = load_dotenv(dotenv_path=env_path, override=True)

print("env file:", env_path)
print("load_dotenv returned:", status)
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning
env file: /coc/scratch/vchopra/learning_env/Deep-Learning/env.env
load_dotenv returned: True


In [89]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "inara" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [90]:
import logging

import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s | %(message)s")

In [91]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [92]:

api_key = os.environ.get("WANDB_API_KEY")
assert api_key, "WANDB_API_KEY was not loaded from env.env"

wandb.finish()
wandb.login(key=api_key, relogin=True)

api = wandb.Api()
print("Logged-in W&B username:", api.viewer.username)
print("Default W&B entity:", api.default_entity)
print("WANDB_API_KEY loaded from environment:", bool(os.getenv("WANDB_API_KEY")))


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /nethome/dgarg39/.netrc


Logged-in W&B username: vedaangchopra
Default W&B entity: vedaangchopra_gatech
WANDB_API_KEY loaded from environment: True


In [93]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [94]:
from model_training.framework.data.post_train.instruction_tuning.interfaces import (
    as_cycling_dataloader,
    create_sft_dataloaders,
    download_instruction_records,
    prepare_sft_data,
    summarize_sft_data,
)

In [95]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 8
0 NVIDIA A40 44.4 GiB
1 NVIDIA A40 44.4 GiB
2 NVIDIA A40 44.4 GiB
3 NVIDIA A40 44.4 GiB
4 NVIDIA A40 44.4 GiB
5 NVIDIA A40 44.4 GiB
6 NVIDIA A40 44.4 GiB
7 NVIDIA A40 44.4 GiB


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [ ]:
file_name = "instruction-data.json"
data_url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data_file = data_dir / "instruction_tuning" / file_name
print(f"[STAGE] load instruction data | {data_file}")
data = download_instruction_records(data_file.parent, file_name, data_url)
print(f"✓ records loaded: {len(data)}")
print("first record:", data[0])

In [99]:
print("last record:", data[-1])

Number of entries: 1100
{'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"} {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


#### Step-2: - Preparing Data for Particular Format and Split it into train and test

In [100]:
# alpaca = AlpacaFormatOld(data)
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [101]:
train_fraction = 0.85
test_fraction = 0.10
print(f"[STAGE] prepare prompt format and splits | train={train_fraction:.0%} test={test_fraction:.0%}")

(935, 110, 55)

In [103]:
prepared_sft = prepare_sft_data(
    data,
    model_name=MODEL_NAME,
    train_fraction=train_fraction,
    test_fraction=test_fraction,
)
sft_data = prepared_sft.prompt_style
tokenizer = prepared_sft.tokenizer
train_data = prepared_sft.train_data
val_data = prepared_sft.validation_data
test_data = prepared_sft.test_data
summary = summarize_sft_data(prepared_sft)
print("prepared record:", summary["prepared_record"])
print(summary["rendered_prompt"])
print(
    f"✓ train={summary['train_examples']} | validation={summary['validation_examples']} "
    f"| test={summary['test_examples']}"
)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

{'prompt': [{'role': 'user', 'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend'}], 'completion': [{'role': 'assistant', 'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-3: - Preparing Tokenizer and DataLoaders

#### Setting up Tokenizer

In [104]:
pad_token_id = tokenizer.pad_token_id
print(f"✓ tokenizer: {MODEL_NAME} | pad token id: {pad_token_id}")

151643

#### Sample Data Prepare

In [ ]:
num_workers = 0
batch_size = 8
max_length = 1024

torch.manual_seed(123)

print(f"[STAGE] tokenize and batch | batch_size={batch_size} max_length={max_length}")
native_loaders = create_sft_dataloaders(
    prepared_sft,
    batch_size=batch_size,
    max_length=max_length,
    num_workers=num_workers,
)
# The native loaders are what Accelerate receives. The wrappers preserve the
# repository's existing next_batch() interface for the single-GPU lesson below.
train_loader = as_cycling_dataloader(native_loaders.train, max_length=max_length)
val_loader = as_cycling_dataloader(native_loaders.validation, max_length=max_length)
test_loader = as_cycling_dataloader(native_loaders.test, max_length=max_length)
print(
    f"✓ batches: train={len(native_loaders.train)} | validation={len(native_loaders.validation)} "
    f"| test={len(native_loaders.test)}"
)

In [ ]:
print("[STAGE] inspect one CPU batch")
sample_input, sample_target = next(iter(native_loaders.train))
print(f"✓ inputs={tuple(sample_input.shape)} | targets={tuple(sample_target.shape)}")
print(f"✓ ignored target tokens: {(sample_target == -100).sum().item()}")

In [ ]:

# inputs_1 = [0, 1, 2, 3, 4]
# inputs_2 = [5, 6]
# inputs_3 = [7, 8, 9]

# batch = (
#     inputs_1,
#     inputs_2,
#     inputs_3
# )
# custom_collate_fn(batch, pad_token_id=pad_token_id)

In [ ]:
# logits_1 = torch.tensor(
#     [[-1.0, 1.0],  # 1st training example
#      [-0.5, 1.5]]  # 2nd training example
# )
# targets_1 = torch.tensor([0, 1])


# loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
# print(loss_1)

# logits_2 = torch.tensor(
#     [[-1.0, 1.0],
#      [-0.5, 1.5],
#      [-0.5, 1.5]]  # New 3rd training example
# )
# targets_2 = torch.tensor([0, 1, 1])

# loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
# print(loss_2)
# targets_3 = torch.tensor([0, 1, -100])

# loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
# print(loss_3)
# print("loss_1 == loss_3:", loss_1 == loss_3)

### Creating, Training and Testing Dataset

### Step-4: - Loading the Model Testing inference

In [ ]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [ ]:
def return_model():
    config =load_model_config(dataset_max_len=1024)
    # MODEL_NAME = 'gpt2'
    model = return_model(MODEL_NAME)

    # model = GPT.from_pretrained(config, "gpt2")
    # print(f"Model Loaded: {model}")
    # model.eval()
    # model = model.to(device)
    
    model.eval()
    text_1 = "Every effort moves you"
    in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                                #   allowed_special = {"<|endoftext|>"}
                                )
    in_tokens = in_tokens.to(next(model.parameters()).device)
    print(in_tokens.shape)  # [1, sequence_length]
    
    out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
    print(token_ids_to_text(out_tokens, tokenizer))

In [ ]:
# import inspect
# inspect.signature(tokenizer.encode)

### SFT-Training

In [ ]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

In [ ]:
from model_training.framework.train.core.train import train_model, TrainComponents
from model_training.framework.train.core.config import TrainConfig

from model_training.framework.train.loss.sft import sft_forward, sft_loss
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [ ]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [ ]:
train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [ ]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

In [ ]:
from torch.utils.data import DataLoader, Subset
from model_training.framework.data.post_train.dataloader import CyclingDataLoader


debug_examples = 80  # 10 batches × batch_size 8

base_loader = train_loader.loader

small_loader = DataLoader(
    Subset(base_loader.dataset, range(debug_examples)),
    batch_size=base_loader.batch_size,
    collate_fn=base_loader.collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=base_loader.num_workers,
)

train_loader_subset = CyclingDataLoader(
    small_loader,
    seq_len=train_loader.seq_len,
)

In [ ]:
train_comp = TrainComponents(optimizer=optimizer, 
                            dataloader=train_loader, 
                            scheduler= cosine_lr, 
                            forward_step=sft_forward,
                            loss_fn=sft_loss
                            )
train_comp

#### Step-7:- Sample Run 

In [ ]:
print("[STAGE] SFT forward and loss")
full_batch = train_comp.dataloader.next_batch()
small_batch = (full_batch[0][:1, :64], full_batch[1][:1, :64])

In [ ]:
logits, labels = train_comp.forward_step(model, small_batch, device)

In [ ]:
loss = train_comp.loss_fn(logits, labels)

In [ ]:
assert logits.shape[:2] == labels.shape
assert loss.ndim == 0 and loss.requires_grad and torch.isfinite(loss)
print(f"✓ logits={tuple(logits.shape)} loss={loss.item():.4f} requires_grad={loss.requires_grad}")

### Final Run

In [ ]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [ ]:
print(train_config.accumulation_steps)

In [ ]:
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger


run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

logger = ExperimentLogger(
    project="sft-experiments",
    log_dir=str(project_root / "execution_results" / "runs" / run_name))

In [ ]:
# %tensorboard --logdir "{project_root}/execution_results/runs"

In [ ]:
model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)

In [ ]:
# text_1 = "What is your name ? "
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
)
in_tokens=in_tokens.to(device)
print(in_tokens.shape)  # [1, sequence_length]


out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))